# Stratified Sampling — Assign strata to intersections

This notebook enriches the merged intersection dataset with stratum labels,
which are used to ensure representative coverage during expert elicitation (pairwise comparison).

Each intersection is assigned to exactly one stratum based on up to four dimensions:
- **Intersection type** — T-junction (street_count=3) vs four-arm (street_count≥4)
- **Historical risk level** — crash frequency from BRON (low / medium / high)
- **VRI presence** — whether a traffic light is within a buffer distance
- **Speed zone** — dominant speed limit of connected road segments (30 / 50+ / onbekend)

In addition, **traffic intensity** is added as a continuous covariate (not a stratum dimension):
- **Traffic intensity** — annual average vehicles/day summed across all approach legs (Fileradar 2024)

Each dimension can be toggled on/off in the config cell below.
Thresholds can be set as quantile breaks (recommended) or absolute breaks.

**Input files:**
- `data/processed/intersections_merged.gpkg` — merged NWB intersections (from notebook 01)
- `data/raw/BRON/*/Ongevallengegevens/ongevallen.txt` — BRON crash records (CSV, latin-1)
- `data/raw/vri_locaties.gpkg` — VRI traffic light locations
- `data/raw/Snelheden_WKD/Snelheden.shp` — national speed limits per road segment
- `data/raw/NWB_intensiteiten/intensiteiten_ProvincieZuid-Holland_2024_fileradar.shp` — Fileradar AI intensity estimates
- `data/raw/Wegvakken.gpkg` — NWB road segments (links junction IDs to segment IDs)
- `data/raw/Buurten/Buurten_vlakken.shp` — Rotterdam neighbourhood polygons (for centrum flag)

**Output file:**
- `data/processed/intersections_stratified.gpkg` — intersections with stratum columns added

**Downstream:** notebook 04 and later notebooks consume `intersections_stratified.gpkg`
instead of `intersections_merged.gpkg`.

## 0. Config

**This is the only cell you need to edit.** All thresholds and toggles live here.

In [1]:
import os

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
PROJECT_DIR = r"C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections"

# Input: final merged intersections from notebook 01
INPUT_INTERSECTIONS = os.path.join(PROJECT_DIR, "data", "processed", "intersections_merged.gpkg")

# Input: BRON crash data root folder.
# Expected structure: BRON/<year>/Ongevallengegevens/ongevallen.txt
# All matching files found under this folder are loaded and combined.
BRON_DIR = os.path.join(PROJECT_DIR, "data", "raw", "BRON")

# Input: VRI traffic light locations
VRI_FILE = os.path.join(PROJECT_DIR, "data", "raw", "vri_locaties.gpkg")

# Input: Snelheden speed limit shapefile (full national dataset, ~1.6M rows)
SNL_FILE = os.path.join(PROJECT_DIR, "data", "raw", "Snelheden_WKD", "Snelheden.shp")

# Input: NWB Wegvakken — links WVK_ID to junction IDs (JTE_ID_BEG / JTE_ID_END)
WVK_FILE = os.path.join(PROJECT_DIR, "data", "processed", "wegvakken_rotterdam_bst_merged.gpkg")

# Input: Fileradar AI intensity estimates per road segment, Zuid-Holland 2024.
# One row per (WVK_ID, RIJRICHTNG) pair; contains INTWERKP50 (vehicles/day, weekday annual avg).
INTENSITY_FILE = os.path.join(PROJECT_DIR, "data", "raw", "NWB_intensiteiten",
                              "intensiteiten_ProvincieZuid-Holland_2024_fileradar.shp")

# Cache: Rotterdam-only Snelheden attributes (no geometry) — created on first run, reused after
SNL_ROT_FILE = os.path.join(PROJECT_DIR, "data", "processed", "snelheden_rotterdam.csv")

# Cache: Rotterdam-only intensity lookup (WVK_ID → INTWERKP50, no geometry).
# Created on first run from the full Zuid-Holland shapefile; delete to force rebuild.
INTENSITY_ROT_FILE = os.path.join(PROJECT_DIR, "data", "processed", "intensiteiten_rotterdam.csv")

# Cache: Rotterdam OSM edges — downloaded once by eda_OSM_voorrangswegen.ipynb, reused here
OSM_CACHE = os.path.join(PROJECT_DIR, "data", "processed", "osm_edges_rotterdam.gpkg")

# Input: Rotterdam neighbourhood polygons — used to flag centrum intersections.
# Not a stratum dimension; used as a proportional sampling constraint downstream.
# Confirmed from eda_buurten.ipynb: CRS is already RD New, 91 polygons, no missing geometry.
BUURTEN_FILE = os.path.join(PROJECT_DIR, "data", "raw", "Buurten", "Buurten_vlakken.shp")

# Column and value that identify the centrum district (confirmed from eda_buurten.ipynb)
CENTRUM_COLUMN = "GEBDNAAM"
CENTRUM_VALUES = ["Rotterdam Centrum"]

# Output: intersections with stratum columns added — used by all downstream notebooks
OUTPUT_INTERSECTIONS = os.path.join(PROJECT_DIR, "data", "processed", "intersections_stratified.gpkg")

CRS_RD = "EPSG:28992"  # RD New — all spatial operations happen in metres

# ---------------------------------------------------------------------------
# Dimension toggles — set False to skip a dimension
# Skipped dimensions get value "_" and are excluded from the stratum label
# ---------------------------------------------------------------------------
USE_INTERSECTION_TYPE = True   # Dim 1: T-junction vs four-arm (from street_count, no extra file needed)
USE_RISK_LEVEL        = False  # Dim 2: crash frequency from BRON (requires BRON_DIR)
USE_VRI               = True   # Dim 3: VRI traffic light within buffer (requires VRI_FILE)
USE_SPEED             = False  # Dim 4: speed zone from Snelheden_WKD (requires SNL_FILE, WVK_FILE)
USE_VOORRANG          = True   # Dim 5: voorrangsweg based on OSM highway classification (requires OSM_CACHE)
USE_INTENSITY         = True   # Covariate: per-segment traffic intensity (Fileradar 2024)
                                # Stored as intensity_max_wvk (busiest connected road) and intensity_bucket.
                                # Not a stratum dimension — used as a covariate in the BT model.
USE_CENTRUM           = True   # Geographic flag: is_centrum=True if intersection is in Rotterdam Centrum
                                # Not a stratum dimension — used as sampling constraint to ensure geographic spread

# ---------------------------------------------------------------------------
# Traffic intensity bucketing
# ---------------------------------------------------------------------------
# Quantile breaks for intensity_bucket classification (applied to intensity_max_wvk).
# P33/P67 produce three equal-frequency classes; thresholds are resolved at run time
# from the Rotterdam intersection distribution and printed for reproducibility.
INTENSITY_QUANTILE_BREAKS = [0.33, 0.67]
INTENSITY_LABELS          = ["laag", "midden", "hoog"]

# ---------------------------------------------------------------------------
# Dimension 1 — Intersection type
# ---------------------------------------------------------------------------
# street_count comes directly from the merged intersection file (notebook 01 output)
# No threshold to tune here.

# ---------------------------------------------------------------------------
# Dimension 2 — Risk level (BRON crash frequency)
# ---------------------------------------------------------------------------
# Year range to filter BRON crashes — only crashes within this range are used.
# All ongevallen.txt files found under BRON_DIR are loaded; the year filter is applied after.
BRON_YEAR_START  = 2020
BRON_YEAR_END    = 2024

# Column names in the BRON dataset (confirmed from eda_BRON.ipynb)
BRON_YEAR_COLUMN = "JAAR_VKL"  # year of the accident
BRON_JTE_COLUMN  = "JTE_ID"    # NWB junction ID — plain integer, directly matchable to NWB

# --- Threshold method ---
# USE_QUANTILE_BREAKS = True:  quantile-based split (recommended — adapts to data distribution)
# USE_QUANTILE_BREAKS = False: absolute break values (fixed thresholds regardless of distribution)
USE_QUANTILE_BREAKS = True

# Quantile breaks — two values produce three classes (low / medium / high)
# Example: [0.50, 0.80] means: bottom 50% = low, 50th–80th percentile = medium, top 20% = high
# Tune these after running eda_BRON.ipynb to inspect the crash count distribution
RISK_QUANTILE_BREAKS = [0.50, 0.80]

# Absolute breaks — used only when USE_QUANTILE_BREAKS = False
# Example: [0, 1] means: 0 crashes = laag, 1 crash = midden, 2+ = hoog
RISK_ABSOLUTE_BREAKS = [0, 1]

# Labels for the three risk classes (low / medium / high) — used in stratum names
RISK_LABELS = ["laag", "midden", "hoog"]

# ---------------------------------------------------------------------------
# Dimension 3 — VRI presence
# ---------------------------------------------------------------------------
# Only full traffic light installations count as VRI — excludes Knipper (flashing amber),
# TWI (pedestrian units), Doseer (ramp metering), and Overig (other).
# Confirmed from eda_vri.ipynb: 292 of 308 records are type "VRI".
VRI_INCLUDE_TYPES = ["VRI"]

# Buffer around each intersection centroid to detect a nearby VRI point (metres)
VRI_BUFFER_M = 30

# ---------------------------------------------------------------------------
# Dimension 4 — Speed zone
# ---------------------------------------------------------------------------
# Speed values treated as the "30" class (low-speed zone).
# All other numeric speeds map to "50+". NVT / NOA / missing → "onbekend".
# Confirmed from eda_snelheden.ipynb: Rotterdam values are 15, 30, 50, 60, 70, 80, 100, 130.
LOW_SPEED_VALUES = {"15", "30"}


# ---------------------------------------------------------------------------
# Dimension 5 — Voorrangsweg (OSM)
# ---------------------------------------------------------------------------
# OSM highway values that indicate a voorrangsweg (priority road) in Rotterdam.
# Based on eda_OSM_voorrangswegen.ipynb: no explicit priority_road tag in OSM Rotterdam;
# highway classification is the only usable proxy.
# tertiary/tertiary_link included after visual spot-check.
VOORRANG_TYPES = {"motorway", "motorway_link", "trunk", "trunk_link",
                  "primary", "primary_link", "secondary", "secondary_link",
                  "tertiary", "tertiary_link"}

# Buffer around intersection centroid to detect nearby voorrangsweg edges (metres)
VOORRANG_BUFFER_M = 20

print("Config loaded.")
print(f"  Intersection type : {'ON' if USE_INTERSECTION_TYPE else 'OFF'}")
print(f"  Risk level (BRON) : {'ON' if USE_RISK_LEVEL else 'OFF'}")
print(f"  VRI               : {'ON' if USE_VRI else 'OFF'}")
print(f"  Speed zone        : {'ON' if USE_SPEED else 'OFF'}")
print(f"  Voorrangsweg      : {'ON' if USE_VOORRANG else 'OFF'}")
print(f"  Traffic intensity : {'ON' if USE_INTENSITY else 'OFF'}  (covariate: intensity_max_wvk + intensity_bucket)")
print(f"  Centrum flag      : {'ON' if USE_CENTRUM else 'OFF'}  (sampling constraint, not a stratum dim)")
if USE_RISK_LEVEL:
    method = f"quantile breaks {RISK_QUANTILE_BREAKS}" if USE_QUANTILE_BREAKS else f"absolute breaks {RISK_ABSOLUTE_BREAKS}"
    print(f"  Risk thresholds   : {method}")
    print(f"  BRON year range   : {BRON_YEAR_START}–{BRON_YEAR_END}")
if USE_VRI:
    print(f"  VRI types kept    : {VRI_INCLUDE_TYPES}")
    print(f"  VRI buffer        : {VRI_BUFFER_M}m")
if USE_SPEED:
    print(f"  Low-speed values  : {LOW_SPEED_VALUES}")
if USE_VOORRANG:
    print(f"  Voorrangsweg types: {sorted(VOORRANG_TYPES)}")
    print(f"  Voorrang buffer   : {VOORRANG_BUFFER_M}m")
if USE_INTENSITY:
    print(f"  Intensity breaks  : P{INTENSITY_QUANTILE_BREAKS[0]*100:.0f} / P{INTENSITY_QUANTILE_BREAKS[1]*100:.0f} → {INTENSITY_LABELS}")

Config loaded.
  Intersection type : ON
  Risk level (BRON) : OFF
  VRI               : ON
  Speed zone        : OFF
  Voorrangsweg      : ON
  Traffic intensity : ON  (covariate: intensity_max_wvk + intensity_bucket)
  Centrum flag      : ON  (sampling constraint, not a stratum dim)
  VRI types kept    : ['VRI']
  VRI buffer        : 30m
  Voorrangsweg types: ['motorway', 'motorway_link', 'primary', 'primary_link', 'secondary', 'secondary_link', 'tertiary', 'tertiary_link', 'trunk', 'trunk_link']
  Voorrang buffer   : 20m
  Intensity breaks  : P33 / P67 → ['laag', 'midden', 'hoog']


## 1. Imports and setup

In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import glob

print("Libraries loaded.")

Libraries loaded.


## 2. Load intersections

In [3]:
# Load the merged intersections produced by notebook 01.
# This is the canonical intersection dataset; we will add stratum columns to it.
intersections = gpd.read_file(INPUT_INTERSECTIONS)

# Ensure the CRS is RD New — all buffer operations require a projected CRS in metres
intersections = intersections.to_crs(CRS_RD)

print(f"Loaded {len(intersections):,} intersections from:")
print(f"  {INPUT_INTERSECTIONS}")
print(f"Columns: {list(intersections.columns)}")

Loaded 4,715 intersections from:
  C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections\data\processed\intersections_merged.gpkg
Columns: ['JTE_ID', 'street_count', 'cluster_size', 'diameter_m', 'ideal_dist_m', 'max_dist_m', 'geometry']


## 3. Dimension 1 — Intersection type

Based on `street_count` (already in the merged file):
- `street_count == 3` → T-splitsing
- `street_count >= 4` → Vierarmig

No extra data source needed.

In [4]:
if USE_INTERSECTION_TYPE:
    # Classify by arm count — T-junctions have exactly 3 connecting roads
    intersections["dim_type"] = np.where(
        intersections["street_count"] == 3,
        "T",    # T-splitsing
        "4+"    # Vierarmig (street_count >= 4)
    )
    print("Dimension 1 — Intersection type:")
    print(intersections["dim_type"].value_counts())
else:
    # Dimension disabled — placeholder value so stratum combining still works
    intersections["dim_type"] = "_"
    print("Dimension 1 skipped (USE_INTERSECTION_TYPE = False).")

Dimension 1 — Intersection type:
dim_type
T     3464
4+    1251
Name: count, dtype: int64


## 4. Dimension 2 — Historical risk level (BRON)

Steps:
1. Load `Ongevallengegevens/ongevallen.txt` as a CSV
2. Filter to the configured year range
3. Keep only crashes with a JTE_ID (junction-located accidents)
4. Count crashes per JTE_ID via a direct tabular join — no spatial join needed
5. Classify counts into low / medium / high using quantile or absolute breaks

**Why direct join instead of spatial join:** BRON stores the NWB junction ID (`JTE_ID`)
directly in the accident record for junction-located crashes. This is more accurate than
buffering centroids, because BRON's own linkage to the road network is used.
Accidents located on a road segment (empty JTE_ID) are excluded from the count.

The resolved threshold values are printed so they are reproducible.

In [5]:
if USE_RISK_LEVEL:
    # --- Find all ongevallen.txt files under BRON_DIR ---
    # Expected structure: BRON/<year>/Ongevallengegevens/ongevallen.txt
    bron_files = glob.glob(
        os.path.join(BRON_DIR, "**", "Ongevallengegevens", "ongevallen.txt"),
        recursive=True
    )

    if not bron_files:
        raise FileNotFoundError(
            f"No ongevallen.txt files found under {BRON_DIR}.\n"
            "Expected structure: BRON/<year>/Ongevallengegevens/ongevallen.txt\n"
            "Or set USE_RISK_LEVEL = False to skip this dimension."
        )

    # Load and concatenate all found files — one per year.
    # BRON files use latin-1 (ISO-8859-1) encoding — covers all byte values 0x00–0xFF.
    parts = [pd.read_csv(f, dtype=str, encoding="latin-1") for f in sorted(bron_files)]
    bron = pd.concat(parts, ignore_index=True)
    print(f"Loaded {len(bron):,} BRON crash records from {len(bron_files)} file(s):")
    for f in sorted(bron_files):
        print(f"  {os.path.relpath(f, PROJECT_DIR)}")

    # --- Filter to configured year range ---
    bron[BRON_YEAR_COLUMN] = pd.to_numeric(bron[BRON_YEAR_COLUMN], errors="coerce")
    bron = bron[
        (bron[BRON_YEAR_COLUMN] >= BRON_YEAR_START) &
        (bron[BRON_YEAR_COLUMN] <= BRON_YEAR_END)
    ]
    print(f"After year filter ({BRON_YEAR_START}–{BRON_YEAR_END}): {len(bron):,} crashes.")

    # --- Keep only junction-located accidents ---
    # BRON records the NWB JTE_ID directly for crashes at a junction.
    # Crashes on a road segment have an empty JTE_ID and cannot be attributed to a junction.
    bron[BRON_JTE_COLUMN] = pd.to_numeric(bron[BRON_JTE_COLUMN], errors="coerce")
    bron_jte = bron.dropna(subset=[BRON_JTE_COLUMN]).copy()
    bron_jte[BRON_JTE_COLUMN] = bron_jte[BRON_JTE_COLUMN].astype(int)
    n_wvk = len(bron) - len(bron_jte)
    print(f"Junction-located crashes (have JTE_ID): {len(bron_jte):,}")
    print(f"Road-segment-located crashes (excluded): {n_wvk:,}")

    # --- Count crashes per junction via direct tabular join ---
    # value_counts() gives crashes per JTE_ID; map onto all NWB intersections
    # so that junctions with 0 BRON crashes get a count of 0 (not NaN)
    crash_counts = bron_jte[BRON_JTE_COLUMN].value_counts()

    # Determine which column holds the NWB junction IDs in the intersections GDF.
    # Notebook 01 saves JTE_ID as the index; gpd.read_file() restores it as a regular column.
    if "JTE_ID" in intersections.columns:
        jte_series = intersections["JTE_ID"].astype(int)
    else:
        jte_series = intersections.index.astype(int)

    intersections["crash_count"] = jte_series.map(crash_counts).fillna(0).astype(int)
    print(f"\nCrash count range: {intersections['crash_count'].min()} – {intersections['crash_count'].max()}")
    print(f"Intersections with 0 crashes: {(intersections['crash_count'] == 0).sum():,}")

    # --- Classify into risk classes ---
    if USE_QUANTILE_BREAKS:
        # Derive break values from the actual distribution.
        # Note: when many intersections have 0 crashes, the lower quantile may also resolve to 0.
        # The resolved values are printed so the classification is reproducible.
        q_values = intersections["crash_count"].quantile(RISK_QUANTILE_BREAKS).tolist()
        unique_edges = sorted(set([-np.inf] + q_values + [np.inf]))
        n_bins = len(unique_edges) - 1
        if n_bins < 3:
            print(f"Warning: quantile breaks {RISK_QUANTILE_BREAKS} collapsed to {n_bins} bins.")
            print("Consider switching to USE_QUANTILE_BREAKS = False with absolute breaks.")
        labels = RISK_LABELS[:n_bins]
        print(f"Quantile breaks {RISK_QUANTILE_BREAKS} → resolved thresholds: {q_values}")
    else:
        unique_edges = sorted(set([-np.inf] + RISK_ABSOLUTE_BREAKS + [np.inf]))
        labels = RISK_LABELS[:len(unique_edges) - 1]
        print(f"Using absolute breaks: {RISK_ABSOLUTE_BREAKS}")

    # pd.cut with resolved edges; right=True means intervals are (lower, upper]
    intersections["dim_risk"] = pd.cut(
        intersections["crash_count"],
        bins=unique_edges,
        labels=labels,
        right=True
    ).astype(str)

    print("\nDimension 2 — Risk level distribution:")
    print(intersections["dim_risk"].value_counts())

else:
    # Dimension disabled — no crash data loaded
    intersections["crash_count"] = np.nan
    intersections["dim_risk"] = "_"
    print("Dimension 2 skipped (USE_RISK_LEVEL = False).")

Dimension 2 skipped (USE_RISK_LEVEL = False).


## 5. Dimension 3 — VRI presence

A VRI (verkeerslicht) is present at an intersection if any VRI point falls within
`VRI_BUFFER_M` metres of the intersection centroid.

VRI intersections have a fundamentally different visual structure and conflict pattern,
making this a strong stratification variable for expert elicitation.

In [6]:
if USE_VRI:
    if not os.path.exists(VRI_FILE):
        raise FileNotFoundError(
            f"VRI file not found: {VRI_FILE}\n"
            "Update VRI_FILE in the config cell, or set USE_VRI = False to skip this dimension."
        )

    # Load VRI point locations and reproject to RD New
    vri = gpd.read_file(VRI_FILE).to_crs(CRS_RD)
    print(f"Loaded {len(vri):,} VRI records.")

    # Filter to configured installation types — excludes Knipper, TWI, Doseer, Overig
    # which are not full intersection traffic lights (confirmed in eda_vri.ipynb)
    vri = vri[vri["Installatietype"].isin(VRI_INCLUDE_TYPES)].copy()
    print(f"After Installatietype filter {VRI_INCLUDE_TYPES}: {len(vri):,} records.")

    # Drop records with missing or empty geometry.
    # notna() alone is not enough — some records have empty (not null) geometries
    # that pass notna() but silently fail spatial operations.
    vri = vri[~vri.geometry.is_empty & vri.geometry.notna()].copy()
    print(f"After dropping null/empty geometry: {len(vri):,} records.")

    # Buffer intersections by VRI_BUFFER_M for the spatial join
    intersections_buffered = intersections.copy()
    intersections_buffered["geometry"] = intersections_buffered.buffer(VRI_BUFFER_M)

    # Spatial join: find which intersection buffers contain at least one VRI point
    vri_joined = gpd.sjoin(
        vri[["geometry"]],
        intersections_buffered[["geometry"]],
        how="inner",
        predicate="within"
    )

    # Intersection indices that have at least one VRI hit
    has_vri = set(vri_joined["index_right"].unique())

    # Assign VRI label — checked against positional index of the intersections GDF
    intersections["dim_vri"] = [
        "VRI" if i in has_vri else "noVRI"
        for i in intersections.index
    ]

    print("\nDimension 3 — VRI presence:")
    print(intersections["dim_vri"].value_counts())

else:
    intersections["dim_vri"] = "_"
    print("Dimension 3 skipped (USE_VRI = False).")

Loaded 308 VRI records.
After Installatietype filter ['VRI']: 292 records.
After dropping null/empty geometry: 278 records.


C:\Users\Thijs\AppData\Local\Temp\ipykernel_768\4134739180.py:20: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  vri = vri[~vri.geometry.is_empty & vri.geometry.notna()].copy()



Dimension 3 — VRI presence:
dim_vri
noVRI    4529
VRI       186
Name: count, dtype: int64


In [7]:
# === DIAGNOSTIC — delete when satisfied ===
import warnings
import geopandas as gpd

vri_diag = gpd.read_file(VRI_FILE).to_crs(CRS_RD)
vri_diag = vri_diag[vri_diag["Installatietype"].isin(VRI_INCLUDE_TYPES)].copy()

# Check how many pass notna() but still have empty geometries
n_notna = vri_diag.geometry.notna().sum()
n_empty = vri_diag.geometry.is_empty.sum()
n_valid = (~vri_diag.geometry.is_empty & vri_diag.geometry.notna()).sum()
print(f"After type filter : {len(vri_diag):,} records")
print(f"  notna()         : {n_notna:,}")
print(f"  empty geom      : {n_empty:,}  <- these pass notna() but fail spatial ops")
print(f"  truly valid     : {n_valid:,}")

# Check CRS of both layers
print(f"\nVRI CRS after reproject : {vri_diag.crs}")
print(f"Intersections CRS       : {intersections.crs}")

# For the valid points only, check matches at various buffer distances
vri_valid = vri_diag[~vri_diag.geometry.is_empty & vri_diag.geometry.notna()].copy()
print(f"\nBuffer distance sensitivity (valid VRI points only):")
for buf in [15, 25, 50, 100]:
    ints_buf = intersections.copy()
    ints_buf["geometry"] = ints_buf.buffer(buf)
    joined = gpd.sjoin(vri_valid[["geometry"]], ints_buf[["geometry"]], how="inner", predicate="within")
    n_ints = joined["index_right"].nunique()
    n_vri  = joined.index.nunique()
    print(f"  {buf:>4}m buffer : {n_ints:>4} intersections matched, {n_vri:>4} VRI points matched")


C:\Users\Thijs\AppData\Local\Temp\ipykernel_768\4138892841.py:9: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  n_notna = vri_diag.geometry.notna().sum()
C:\Users\Thijs\AppData\Local\Temp\ipykernel_768\4138892841.py:11: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()'

After type filter : 292 records
  notna()         : 292
  empty geom      : 14  <- these pass notna() but fail spatial ops
  truly valid     : 278

VRI CRS after reproject : EPSG:28992
Intersections CRS       : EPSG:28992

Buffer distance sensitivity (valid VRI points only):
    15m buffer :  138 intersections matched,  137 VRI points matched
    25m buffer :  175 intersections matched,  155 VRI points matched
    50m buffer :  250 intersections matched,  166 VRI points matched
   100m buffer :  558 intersections matched,  177 VRI points matched


## 6. Dimension 4 — Speed zone

The dominant speed limit of road segments connected to each intersection is used
to distinguish low-speed residential zones (30 km/h) from standard urban roads (50+ km/h).

**Join strategy** (confirmed in eda_snelheden.ipynb):
1. Load `Wegvakken.gpkg` — maps each `WVK_ID` to its begin/end junction (`JTE_ID_BEG`, `JTE_ID_END`)
2. Build a junction → connected segments table
3. Look up `MAXSHD` (speed limit) for each segment from `Snelheden.shp`
4. Per junction, take the **mode** of meaningful speeds (excluding NVT / NOA)

**Classes:**
- `30` — dominant speed is 30 km/h or lower (residential/low-speed zone)
- `50+` — dominant speed is 50 km/h or higher (standard or higher urban speed)
- `50+` also covers NVT / NOA / no data (e.g. pedestrian paths, unclassified segments) — treated as standard urban speed

In [8]:
if USE_SPEED:
    # --- Load Snelheden (speed limit per road segment) ---
    # Use the Rotterdam cache if it already exists (created by eda_snelheden.ipynb or a prior run).
    # On first run, load the full national shapefile (~1.6M rows), filter to Rotterdam, and save cache.
    if os.path.exists(SNL_ROT_FILE):
        snl_rot = pd.read_csv(SNL_ROT_FILE, dtype=str)
        print(f"Loaded Snelheden from cache: {len(snl_rot):,} Rotterdam rows")
    else:
        print("Snelheden cache not found — loading full Snelheden.shp (takes a moment)...")
        import geopandas as _gpd  # alias to avoid shadowing the top-level gpd import
        snl_full = _gpd.read_file(SNL_FILE)
        snl_rot = snl_full[snl_full["GME_NAAM"] == "Rotterdam"].copy()
        snl_rot.drop(columns=["geometry"]).to_csv(SNL_ROT_FILE, index=False)
        print(f"Filtered to Rotterdam: {len(snl_rot):,} rows — saved cache to {os.path.relpath(SNL_ROT_FILE, PROJECT_DIR)}")

    # --- Load Wegvakken to build the junction → segment mapping ---
    # Use the Rotterdam BST-merged geopackage from notebook 01 — already filtered
    # to Rotterdam gemeente roads, so much smaller than the full national WVK_FILE.
    print("Loading Wegvakken (Rotterdam BST-merged)...")
    wvk = gpd.read_file(
        os.path.join(PROJECT_DIR, "data", "processed", "wegvakken_rotterdam_bst_merged.gpkg"),
        ignore_geometry=True
    )

    # Build long-format table: one row per (junction, connected segment) pair.
    # Each segment appears twice — once for JTE_ID_BEG, once for JTE_ID_END.
    wvk_b = wvk[["JTE_ID_BEG", "WVK_ID"]].rename(columns={"JTE_ID_BEG": "JTE_ID"})
    wvk_e = wvk[["JTE_ID_END", "WVK_ID"]].rename(columns={"JTE_ID_END": "JTE_ID"})
    jte_wvk = pd.concat([wvk_b, wvk_e], ignore_index=True)
    jte_wvk = jte_wvk.dropna(subset=["JTE_ID"])
    jte_wvk["JTE_ID"] = jte_wvk["JTE_ID"].astype(int)

    # Filter to only the JTE_IDs present in our intersection dataset — reduces memory use
    int_jte = set(intersections["JTE_ID"].astype(int))
    matched = jte_wvk[jte_wvk["JTE_ID"].isin(int_jte)].copy()
    print(f"Intersections matched in Wegvakken: {matched['JTE_ID'].nunique():,} / {len(int_jte):,}")

    # --- Look up MAXSHD for each connected segment ---
    # snl_rot was loaded with dtype=str, so WVK_ID keys are strings — cast matched to match
    speed_lookup = snl_rot.set_index("WVK_ID")["MAXSHD"].to_dict()
    matched["MAXSHD"] = matched["WVK_ID"].astype(str).map(speed_lookup)

    # --- Aggregate per junction: mode of meaningful speeds ---
    # Exclude NVT (not applicable) and NOA (not determinable) so they don't override real values.
    # If no meaningful speeds exist for a junction, fall back to "NVT".
    EXCLUDE_CODES = {"NVT", "NOA"}

    def dominant_speed(series):
        meaningful = series[~series.isin(EXCLUDE_CODES) & series.notna()]
        if len(meaningful) == 0:
            return "NVT"
        return meaningful.mode().iloc[0]

    speed_per_junction = matched.groupby("JTE_ID")["MAXSHD"].agg(
        dominant_speed=dominant_speed
    ).reset_index()

    # Merge dominant speed onto intersections GeoDataFrame
    intersections["JTE_ID_int"] = intersections["JTE_ID"].astype(int)
    intersections = intersections.merge(
        speed_per_junction.rename(columns={"JTE_ID": "JTE_ID_int"}),
        on="JTE_ID_int",
        how="left"
    ).drop(columns=["JTE_ID_int"])

    # --- Classify into speed strata ---
    # LOW_SPEED_VALUES → "30", all other numeric speeds → "50+", NVT/NOA/missing → "onbekend"
    def classify_speed(val):
        # NVT/NOA/missing are treated as standard urban speed (50+)
        if str(val) in LOW_SPEED_VALUES:
            return "30"
        return "50+"

    intersections["dim_speed"] = intersections["dominant_speed"].apply(classify_speed)

    print("\nDimension 4 — Speed zone:")
    print(intersections["dim_speed"].value_counts())

else:
    # Dimension disabled — placeholder so stratum combining still works
    intersections["dim_speed"] = "_"
    print("Dimension 4 skipped (USE_SPEED = False).")

Dimension 4 skipped (USE_SPEED = False).


## 7. Dimension 5 — Voorrangsweg (OSM)

An intersection is classified as `voorrang` if any road edge within a buffer
is tagged as a voorrangsweg in OSM. The result is combined with VRI presence
into a single `dim_priority` column in the cell below:

- **VRI** — intersection has a traffic light (regardless of road class)
- **voorrang** — no traffic light, but on a priority road
- **geen_voorrang** — no traffic light, not on a priority road

OSM has no `priority_road` tag in Rotterdam; the proxy is the `highway` value.
`tertiary` roads are included after visual inspection.

In [9]:
if USE_VOORRANG:
    if not os.path.exists(OSM_CACHE):
        raise FileNotFoundError(
            f"OSM edge cache not found: {OSM_CACHE}"
            "Run eda_OSM_voorrangswegen.ipynb first to download and cache the road network."
        )

    # Load cached OSM edges — already in RD New from the EDA notebook
    osm_edges = gpd.read_file(OSM_CACHE)
    if osm_edges.crs != CRS_RD:
        osm_edges = osm_edges.to_crs(CRS_RD)
    print(f"Loaded {len(osm_edges):,} OSM edges from cache.")

    # Classify each edge as voorrangsweg based on its highway value.
    # highway can be a list (multi-tagged edges) — check if any value is in VOORRANG_TYPES.
    def is_voorrangsweg(hw_val):
        if isinstance(hw_val, list):
            return any(v in VOORRANG_TYPES for v in hw_val)
        return hw_val in VOORRANG_TYPES

    osm_edges["voorrangsweg"] = osm_edges["highway"].apply(is_voorrangsweg)
    print(f"Voorrangsweg edges: {osm_edges['voorrangsweg'].sum():,} / {len(osm_edges):,}")

    # Buffer intersections to catch connected road edges within VOORRANG_BUFFER_M.
    # This mirrors how VRI presence is detected — look at connected segments near the centroid.
    ints_buf = intersections[["JTE_ID", "geometry"]].copy()
    ints_buf["geometry"] = ints_buf.buffer(VOORRANG_BUFFER_M)

    # Spatial join: find which intersection buffers intersect a voorrangsweg edge
    vw_joined = gpd.sjoin(
        osm_edges[osm_edges["voorrangsweg"]][["geometry"]],
        ints_buf,
        how="inner",
        predicate="intersects"
    )

    # An intersection is voorrangsweg if ANY touching edge is classified as such
    has_voorrang = set(vw_joined["JTE_ID"].unique())
    intersections["dim_voorrang"] = intersections["JTE_ID"].apply(
        lambda jte: "voorrang" if jte in has_voorrang else "geen_voorrang"
    )

    print("Dimension 5 — Voorrangsweg:")
    print(intersections["dim_voorrang"].value_counts())
    print(f"  {(intersections['dim_voorrang'] == 'voorrang').mean()*100:.1f}% of intersections on a voorrangsweg")
else:
    # Dimension disabled — set placeholder so stratum combining still works
    intersections["dim_voorrang"] = "_"
    print("Dimension 5 skipped (USE_VOORRANG = False).")


Loaded 25,577 OSM edges from cache.
Voorrangsweg edges: 5,507 / 25,577
Dimension 5 — Voorrangsweg:
dim_voorrang
geen_voorrang    3595
voorrang         1120
Name: count, dtype: int64
  23.8% of intersections on a voorrangsweg


In [10]:
# Combine VRI presence and voorrangsweg status into a single hierarchical dimension.
#
# Priority order: VRI > voorrangsweg > geen_voorrang
#   - "VRI"          : intersection has a traffic light (regardless of road class)
#   - "voorrang"     : no traffic light, but on a priority road (OSM highway classification)
#   - "geen_voorrang": no traffic light, not on a priority road
#
# has_vri uses positional index (from the spatial join result index_right).
# has_voorrang uses JTE_ID (from the OSM edge spatial join).
if USE_VRI or USE_VOORRANG:
    intersections["dim_priority"] = [
        "VRI"          if idx in has_vri else
        "voorrang"     if jte in has_voorrang else
        "geen_voorrang"
        for idx, jte in zip(intersections.index, intersections["JTE_ID"])
    ]

    print("Combined priority dimension (dim_priority):")
    print(intersections["dim_priority"].value_counts())
    print(f"Active sources: VRI={'ON' if USE_VRI else 'OFF'}, Voorrangsweg={'ON' if USE_VOORRANG else 'OFF'}")
else:
    intersections["dim_priority"] = "_"
    print("Priority dimension skipped (USE_VRI = False and USE_VOORRANG = False).")


Combined priority dimension (dim_priority):
dim_priority
geen_voorrang    3593
voorrang          936
VRI               186
Name: count, dtype: int64
Active sources: VRI=ON, Voorrangsweg=ON


## 6b. Traffic intensity (covariate)

Traffic intensity is added as a **continuous covariate**, not a stratification dimension.
It is used as an explanatory variable in the BT model after the survey, not for sampling.

**Source:** Fileradar AI model, annual average 2024 (NDW).
Three percentile columns available; `INTWERKP50` (median estimate) is used as the point value.

**What is stored per intersection:**
- `intensity_max_wvk` — two-way daily flow (vehicles/day) of the *busiest* connected road segment.
  Summing across all approach legs was rejected: the total is proportional to arm count and does not
  reflect the actual traffic environment at the junction.
- `intensity_bucket` — three-class label (`laag` / `midden` / `hoog`) derived from the P33/P67
  quantile breaks of `intensity_max_wvk` across all Rotterdam intersections with data.

**Join strategy** (confirmed in eda_nwb_intensiteiten.ipynb):
1. Load shapefile attributes only (no geometry needed — tabular join via `WVK_ID`)
2. Filter to Rotterdam WVK_IDs; exclude `METHODE='Geen'` (sentinel `-1`)
3. Sum H + T rows per `WVK_ID` → two-way daily flow per road segment
4. Build junction → WVK_ID table from `JTE_ID_BEG` / `JTE_ID_END` in the intensity file
5. Take the **max** two-way flow across all connected segments per junction → `intensity_max_wvk`
6. Apply P33/P67 quantile breaks → `intensity_bucket`

The full Zuid-Holland shapefile (61 MB) is loaded only once; a Rotterdam-only CSV cache
is saved to `data/processed/intensiteiten_rotterdam.csv` for all subsequent runs.

In [11]:
if USE_INTENSITY:
    if not os.path.exists(INTENSITY_FILE):
        raise FileNotFoundError(
            f"Intensity file not found: {INTENSITY_FILE}\n"
            "Place the Fileradar shapefile under data/raw/NWB_intensiteiten/\n"
            "or set USE_INTENSITY = False to skip this step."
        )

    # Use the Rotterdam cache if it exists to avoid reloading the full Zuid-Holland file (~61 MB)
    if os.path.exists(INTENSITY_ROT_FILE):
        int_rot = pd.read_csv(INTENSITY_ROT_FILE)
        print(f"Loaded intensity from cache: {len(int_rot):,} rows")
    else:
        print("Intensity cache not found — loading full shapefile (61 MB, takes a moment)...")
        # ignore_geometry=True skips the WGS84 geometry — only attributes are needed for a tabular join
        int_full = gpd.read_file(INTENSITY_FILE, ignore_geometry=True)

        # Filter to Rotterdam: keep only WVK_IDs present in our BST-merged wegvakken
        wvk_rot = gpd.read_file(WVK_FILE, ignore_geometry=True)
        rotterdam_wvk_ids = set(wvk_rot["WVK_ID"].astype(int))
        int_rot = int_full[int_full["WVK_ID"].astype(int).isin(rotterdam_wvk_ids)].copy()

        int_rot.to_csv(INTENSITY_ROT_FILE, index=False)
        print(f"Filtered to Rotterdam: {len(int_rot):,} rows — cache saved to {os.path.relpath(INTENSITY_ROT_FILE, PROJECT_DIR)}")

    # Exclude segments where Fileradar could not produce an estimate.
    # These rows have METHODE='Geen' and sentinel value -1 for all percentile columns.
    int_rot = int_rot[int_rot["METHODE"] != "Geen"].copy()
    int_rot["WVK_ID_int"] = int_rot["WVK_ID"].astype(int)

    # Sum H (heen) and T (terug) rows per WVK_ID to get total two-way daily flow per segment.
    # Every segment appears once per direction; summing gives the road's total two-way volume.
    wvk_intensity = (
        int_rot.groupby("WVK_ID_int")["INTWERKP50"]
        .sum()
        .rename("intensity_twoway")
    )

    # Build junction → WVK_ID mapping using JTE_ID_BEG and JTE_ID_END directly from the
    # intensity file — avoids a separate load of wegvakken_rotterdam_bst_merged.gpkg.
    jte_beg = int_rot[["JTE_ID_BEG", "WVK_ID_int"]].rename(columns={"JTE_ID_BEG": "JTE_ID"})
    jte_end = int_rot[["JTE_ID_END", "WVK_ID_int"]].rename(columns={"JTE_ID_END": "JTE_ID"})
    jte_wvk_int = pd.concat([jte_beg, jte_end]).drop_duplicates()
    jte_wvk_int["JTE_ID"] = jte_wvk_int["JTE_ID"].astype(int)

    # Look up two-way intensity for each connected segment
    jte_wvk_int["intensity"] = jte_wvk_int["WVK_ID_int"].map(wvk_intensity)

    # Take the max two-way intensity across all connected segments per junction.
    # Summing across approach legs inflates the value proportionally to arm count, which is
    # not meaningful as an intensity metric; the busiest connected road is a better proxy
    # for the traffic environment the intersection is embedded in.
    intensity_per_jte = (
        jte_wvk_int.dropna(subset=["intensity"])
        .groupby("JTE_ID")["intensity"]
        .max()
        .rename("intensity_max_wvk")
        .reset_index()
    )

    # Compute p33/p67 thresholds from the valid-junction distribution and assign bucket labels.
    # Thresholds are derived at run time so they adapt to the Rotterdam subset; printed for
    # reproducibility.
    valid_intensity = intensity_per_jte["intensity_max_wvk"]
    p33 = valid_intensity.quantile(INTENSITY_QUANTILE_BREAKS[0])
    p67 = valid_intensity.quantile(INTENSITY_QUANTILE_BREAKS[1])

    def _classify_intensity(val):
        if pd.isna(val):    return np.nan
        if val < p33:       return INTENSITY_LABELS[0]   # laag
        if val <= p67:      return INTENSITY_LABELS[1]   # midden
        return                     INTENSITY_LABELS[2]   # hoog

    intensity_per_jte["intensity_bucket"] = intensity_per_jte["intensity_max_wvk"].apply(_classify_intensity)

    print(f"Resolved thresholds (n={len(valid_intensity):,} valid junctions):")
    print(f"  P{INTENSITY_QUANTILE_BREAKS[0]*100:.0f} = {p33:,.0f} vrtg/dag  |  "
          f"P{INTENSITY_QUANTILE_BREAKS[1]*100:.0f} = {p67:,.0f} vrtg/dag  (busiest connected segment)")
    print("Bucket distribution:")
    print(intensity_per_jte["intensity_bucket"].value_counts().sort_index().to_string())

    # Merge onto intersections GeoDataFrame
    intersections["JTE_ID_int"] = intersections["JTE_ID"].astype(int)
    intersections = intersections.merge(
        intensity_per_jte.rename(columns={"JTE_ID": "JTE_ID_int"}),
        on="JTE_ID_int",
        how="left"
    ).drop(columns=["JTE_ID_int"])

    n_matched = intersections["intensity_max_wvk"].notna().sum()
    print(f"\nIntersections with intensity data : {n_matched:,} / {len(intersections):,}  ({n_matched/len(intersections)*100:.1f}%)")
    print(f"intensity_max_wvk stats (vehicles/day, busiest connected segment):")
    print(intersections["intensity_max_wvk"].describe().round(0))

else:
    # Dimension disabled — NaN placeholders so both columns are always present downstream
    intersections["intensity_max_wvk"] = np.nan
    intersections["intensity_bucket"]  = np.nan
    print("Traffic intensity skipped (USE_INTENSITY = False).")

C:\Users\Thijs\AppData\Local\Temp\ipykernel_768\4099528405.py:11: DtypeWarning: Columns (0: WEGNUMMER, 1: WEGNR_HMP) have mixed types. Specify dtype option on import or set low_memory=False.
  int_rot = pd.read_csv(INTENSITY_ROT_FILE)


Loaded intensity from cache: 37,306 rows
Resolved thresholds (n=19,002 valid junctions):
  P33 = 400 vrtg/dag  |  P67 = 2,750 vrtg/dag  (busiest connected segment)
Bucket distribution:
intensity_bucket
hoog      6246
laag      5743
midden    7013

Intersections with intensity data : 4,565 / 4,715  (96.8%)
intensity_max_wvk stats (vehicles/day, busiest connected segment):
count     4565.0
mean      3095.0
std       4049.0
min          0.0
25%        450.0
50%       1200.0
75%       4100.0
max      28550.0
Name: intensity_max_wvk, dtype: float64


## 7. Combine dimensions into a single stratum label

Active dimensions are joined with `_` to form the stratum label, e.g. `T_laag_VRI_50+`.
Disabled dimensions (value `"_"`) are excluded from the label so the label stays readable.

In [12]:
# Collect the active dimension columns (those not set to the disabled placeholder "_")
dim_columns = []
if USE_INTERSECTION_TYPE: dim_columns.append("dim_type")
if USE_RISK_LEVEL:        dim_columns.append("dim_risk")
if USE_SPEED:             dim_columns.append("dim_speed")
if USE_VRI or USE_VOORRANG: dim_columns.append("dim_priority")  # VRI + voorrangsweg combined

if not dim_columns:
    raise ValueError("All dimensions are disabled. Enable at least one dimension in the config.")

# Concatenate dimension values into a single stratum label per intersection
# Example: dim_type="T", dim_risk="laag", dim_vri="VRI", dim_speed="30" → stratum="T_laag_VRI_30"
intersections["stratum"] = intersections[dim_columns].apply(
    lambda row: "_".join(row.values.astype(str)), axis=1
)

print(f"Active dimensions: {dim_columns}")
print(f"Number of strata: {intersections['stratum'].nunique()}")
print()

Active dimensions: ['dim_type', 'dim_priority']
Number of strata: 6



## 8. Quality check — stratum distribution

Print the count per stratum. Flag any stratum with fewer than 5 intersections —
these are too small to sample from reliably and may need to be merged with a
neighbouring stratum, or the threshold tuned.

In [13]:
# Minimum cell size for reliable Bradley-Terry estimation (empirical lower bound)
MIN_STRATUM_SIZE = 5

stratum_counts = intersections["stratum"].value_counts().sort_index()

print("Stratum distribution:")
print("-" * 35)
for stratum, count in stratum_counts.items():
    flag = "  ⚠ too small" if count < MIN_STRATUM_SIZE else ""
    print(f"  {stratum:<20} {count:>4}{flag}")
print("-" * 35)
print(f"  Total: {len(intersections):,} intersections across {len(stratum_counts)} strata")

# Warn if any stratum is below the minimum
small = stratum_counts[stratum_counts < MIN_STRATUM_SIZE]
if len(small) > 0:
    print(f"\nWarning: {len(small)} stratum/strata below minimum size ({MIN_STRATUM_SIZE}).")
    print("Consider adjusting thresholds in the config or merging small strata.")
else:
    print("\nAll strata meet the minimum size requirement.")

Stratum distribution:
-----------------------------------
  4+_VRI                168
  4+_geen_voorrang      699
  4+_voorrang           384
  T_VRI                  18
  T_geen_voorrang      2894
  T_voorrang            552
-----------------------------------
  Total: 4,715 intersections across 6 strata

All strata meet the minimum size requirement.


## 9. Geographic constraint — centrum flag

Add `is_centrum` to each intersection: `True` if it falls within the Rotterdam Centrum
district (`GEBDNAAM = "Rotterdam Centrum"`), `False` otherwise.

**This is NOT a stratum dimension.** Adding it as a stratum would double the number of
cells (6 → 12) and halve per-cell sample size, which hurts BT estimation power.
Instead, `is_centrum` is saved as a column and used as a **proportional sampling constraint**
in the sampling step: within each stratum, draw ~9% of the sample from centrum intersections
(matching the natural share confirmed in eda_buurten.ipynb).

All strata contain centrum intersections (7.7%–24.4%), so the constraint is always satisfiable.

In [14]:
if USE_CENTRUM:
    # Load buurten — already in RD New (confirmed from eda_buurten.ipynb), no reprojection needed
    buurten = gpd.read_file(BUURTEN_FILE)
    if str(buurten.crs) != CRS_RD:
        buurten = buurten.to_crs(CRS_RD)

    # Keep only the centrum polygons to simplify the spatial join
    centrum_polygons = buurten[buurten[CENTRUM_COLUMN].isin(CENTRUM_VALUES)][["geometry"]].copy()
    print(f"Loaded {len(buurten)} buurten, {len(centrum_polygons)} identified as centrum.")

    # Spatial join: flag each intersection that falls within a centrum polygon.
    # 'within' is correct here — intersection centroids are Point geometries.
    joined = gpd.sjoin(
        intersections[["JTE_ID", "geometry"]],
        centrum_polygons,
        how="left",
        predicate="within"
    )
    # index_right is NaN for intersections outside all centrum polygons
    centrum_jte_ids = set(joined[joined["index_right"].notna()]["JTE_ID"])
    intersections["is_centrum"] = intersections["JTE_ID"].isin(centrum_jte_ids)

    n_centrum = intersections["is_centrum"].sum()
    print(f"Centrum intersections : {n_centrum} ({n_centrum / len(intersections) * 100:.1f}%)")
    print(f"Niet-centrum          : {len(intersections) - n_centrum}")
    print()
    print("Centrum count per stratum (reference for proportional sampling):")
    ct = intersections.groupby("stratum")["is_centrum"].agg(["sum", "count"])
    ct["pct_centrum"] = (ct["sum"] / ct["count"] * 100).round(1)
    ct.columns = ["centrum", "total", "pct_centrum"]
    print(ct.to_string())
else:
    # Disabled — NaN placeholder so the column is always present in the output file
    intersections["is_centrum"] = pd.NA
    print("Centrum flag skipped (USE_CENTRUM = False).")

Loaded 91 buurten, 6 identified as centrum.
Centrum intersections : 430 (9.1%)
Niet-centrum          : 4285

Centrum count per stratum (reference for proportional sampling):
                  centrum  total  pct_centrum
stratum                                      
4+_VRI                 41    168         24.4
4+_geen_voorrang       67    699          9.6
4+_voorrang            34    384          8.9
T_VRI                   3     18         16.7
T_geen_voorrang       222   2894          7.7
T_voorrang             63    552         11.4


## 10. Save output

Saves to `intersections_stratified.gpkg` — a strict superset of the input file.
All original columns are preserved; the stratum columns (`dim_*`, `stratum`, `crash_count`,
`dominant_speed`, `intensity_sum`, `is_centrum`) are added on top.
Downstream notebooks use this file instead of `intersections_merged.gpkg`.

In [15]:
# Save enriched intersections — preserves all original columns plus stratum columns
intersections.to_file(OUTPUT_INTERSECTIONS, driver="GPKG")

print(f"Saved {len(intersections):,} intersections to:")
print(f"  {OUTPUT_INTERSECTIONS}")
print()
print("New columns added:")
new_cols = [c for c in intersections.columns if c.startswith("dim_") or c in (
    "stratum", "crash_count", "intensity_max_wvk", "intensity_bucket", "is_centrum"
)]
for col in new_cols:
    print(f"  {col}")
print()
print("You can now run notebook 04 — it should consume intersections_stratified.gpkg.")

Saved 4,715 intersections to:
  C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections\data\processed\intersections_stratified.gpkg

New columns added:
  dim_type
  crash_count
  dim_risk
  dim_vri
  dim_speed
  dim_voorrang
  dim_priority
  intensity_max_wvk
  intensity_bucket
  stratum
  is_centrum

You can now run notebook 04 — it should consume intersections_stratified.gpkg.
